# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name: Jesse Wang
Date: 2026-08-17

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/wangjing/bootcamp_Jesse_Wang/homework/homework4

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? False


## Helpers (use or modify)

In [4]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k, v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    dtypes = {c: str(df[c].dtype) for c in required if c in df.columns}
    return {
        'missing': missing,
        'shape': df.shape,
        'na_total': int(df.isna().sum().sum()),
        'dtypes': dtypes,
    }

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [5]:
SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
print('Using Alpha Vantage:', USE_ALPHA)

df_api = None
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {
        'function': 'TIME_SERIES_DAILY',   # raw close (adjusted close is a paid field)
        'symbol': SYMBOL,
        'outputsize': 'compact',
        'apikey': os.getenv('ALPHAVANTAGE_API_KEY'),
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    # Alpha Vantage replies HTTP 200 with a prose error blob (not a status code)
    # when the free tier's ~25/day cap is hit or a field moved behind a paywall,
    # so don't trust the status code alone - look for the series key.
    key = [k for k in js if 'Time Series' in k]
    if key:
        df_api = (pd.DataFrame(js[key[0]]).T
                  .rename_axis('date').reset_index()
                  [['date', '4. close']].rename(columns={'4. close': 'close'}))
        df_api['date'] = pd.to_datetime(df_api['date'])
        df_api['close'] = pd.to_numeric(df_api['close'])
    else:
        print('Alpha Vantage returned no series (rate limit / premium field?):',
              str(list(js.values())[0])[:120])
        print('Falling back to yfinance.')

if df_api is None:
    import yfinance as yf
    df_api = (yf.download(SYMBOL, period='6mo', interval='1d',
                          auto_adjust=False, progress=False,
                          multi_level_index=False)
              .reset_index()[['Date', 'Close']])
    df_api.columns = ['date', 'close']

df_api = df_api.sort_values('date').reset_index(drop=True)
v_api = validate(df_api, ['date', 'close'])
v_api

Using Alpha Vantage: False


{'missing': [],
 'shape': (125, 2),
 'na_total': 0,
 'dtypes': {'date': 'datetime64[s]', 'close': 'float64'}}

In [6]:
_ = save_csv(df_api, prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data/raw/api_source-yfinance_symbol-AAPL_20260817-175011.csv


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [7]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {'User-Agent': 'AFE-Homework/1.0'}

df_scrape = None
try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    # Prefer the stable id; fall back to the first wikitable if the id changes.
    table = soup.find('table', id='constituents') or soup.find('table', class_='wikitable')
    if table is None:
        raise RuntimeError('no <table> found on the page')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th', 'td'])]
            for tr in table.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)
    # Parse dtypes: CIK is an integer id, "Date added" is a date; the rest stays text.
    if 'CIK' in df_scrape.columns:
        df_scrape['CIK'] = pd.to_numeric(df_scrape['CIK'], errors='coerce').astype('Int64')
    if 'Date added' in df_scrape.columns:
        df_scrape['Date added'] = pd.to_datetime(df_scrape['Date added'], errors='coerce')
except Exception as e:
    print('Scrape failed, using inline demo table:', e)
    html = ('<table><tr><th>Ticker</th><th>Price</th></tr>'
            '<tr><td>AAA</td><td>101.2</td></tr>'
            '<tr><td>BBB</td><td>98.7</td></tr></table>')
    soup = BeautifulSoup(html, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th', 'td'])]
            for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)
    if 'Price' in df_scrape.columns:
        df_scrape['Price'] = pd.to_numeric(df_scrape['Price'], errors='coerce')

v_scrape = validate(df_scrape, list(df_scrape.columns))
v_scrape

{'missing': [],
 'shape': (503, 8),
 'na_total': 0,
 'dtypes': {'Symbol': 'str',
  'Security': 'str',
  'GICSSector': 'str',
  'GICS Sub-Industry': 'str',
  'Headquarters Location': 'str',
  'Date added': 'datetime64[us]',
  'CIK': 'Int64',
  'Founded': 'str'}}

In [8]:
_ = save_csv(df_scrape, prefix='scrape', site='wikipedia', table='sp500')

Saved data/raw/scrape_site-wikipedia_table-sp500_20260817-175011.csv


## Documentation

### API Source
- **Endpoint:** Alpha Vantage `TIME_SERIES_DAILY` (fallback: `yfinance.download`)
- **URL / params:** `https://www.alphavantage.co/query` with
  `function=TIME_SERIES_DAILY`, `symbol=AAPL`, `outputsize=compact`, `apikey=<from .env>`.
- **Key handling:** the key is read from `.env` via `python-dotenv` and never hard-coded.
  If it is missing, or Alpha Vantage answers HTTP 200 with an error blob (free tier ≈25
  calls/day), the code falls back to `yfinance`.
- **Field choice:** raw `close` (not `adjusted close`), because adjusted close is a paid
  field at Alpha Vantage.

### Scrape Source
- **URL:** `https://en.wikipedia.org/wiki/List_of_S%26P_500_companies` (public, permitted table).
- **Table:** `table#constituents` (S&P 500 constituents), with a fallback to the first
  `table.wikitable` if the id changes.
- **Parsing:** BeautifulSoup → `th`/`td` text rows → DataFrame; `CIK` → integer,
  `Date added` → datetime, remaining columns kept as text.

### Validation
- Required columns present (`date`, `close` for the API; every parsed column for the scrape).
- Shape, total NA count, and per-column dtypes are reported by `validate()`.

### Assumptions & Risks
- **Rate limits:** Alpha Vantage free tier ≈25 calls/day → handled via the yfinance fallback.
- **Selector fragility:** Wikipedia table markup can change; the `id`-first / `class`-fallback
  selector plus the inline demo-table `except` path keep the scrape resilient.
- **Schema changes:** a vendor moving a column behind a paywall (e.g. adjusted close) breaks
  pipelines that assume it exists — the code selects the raw `close` and checks for the series.
- **Timestamped filenames:** `save_csv` embeds a `YYYYMMDD-HHMMSS` stamp, so reruns produce
  fresh raw files instead of overwriting.

### Secrets
- `.env` (holding `ALPHAVANTAGE_API_KEY`) is **gitignored** here and at the repo root;
  only `.env.example` (no real key) is committed.